# 🧾 Agente de Facturación Electrónica con LangChain

**Caso de uso:** Automatización de lectura de facturas electrónicas desde correo IMAP.

**Arquitectura:**
```
┌────────────────────────────────────────────────────┐
│         LangGraph Agent (ReAct)                    │
│               LLM: GPT-4o                          │
├────────────────────────────────────────────────────┤
│  Tools:                                            │
│  1. leer_correos    → IMAP, correos no leídos     │
│  2. extraer_adjuntos → ZIP → PDF + XML            │
│  3. parsear_factura  → XML → datos estructurados  │
├────────────────────────────────────────────────────┤
│  Output: DataFrame (tabla) + archivos en disco     │
└────────────────────────────────────────────────────┘
```

**Componentes LangChain:**
- `ChatOpenAI` (GPT-4o)
- `ChatPromptTemplate` (system + human)
- `@tool` decorators (Tools personalizadas)
- `langgraph.prebuilt.create_react_agent` — Agente ReAct

## 1. Instalación de dependencias

In [ ]:
!pip install -q langchain-openai langgraph langchain-core pandas

## 2. Configuración

In [ ]:
import os
from google.colab import userdata

# Configuración del LLM (desde Secrets de Colab)
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Configuración del correo IMAP (desde Secrets de Colab)
IMAP_HOST = "mail.suprokom.com"
IMAP_PORT = 993
IMAP_USER = "facturacion@suprokom.com"
IMAP_PASSWORD = userdata.get("IMAP_PASSWORD")

# Directorio para guardar archivos extraídos
OUTPUT_DIR = "facturas_extraidas"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Configuración lista")

## 3. Imports principales

In [ ]:
import imaplib
import email
from email.header import decode_header
import zipfile
import io
import os
import xml.etree.ElementTree as ET
import pandas as pd
from datetime import datetime

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

print("✅ Imports cargados")

## 4. Definición de Tools (Herramientas del Agente)

### 4.1 Tool: Leer correos no leídos

In [ ]:
@tool
def leer_correos(cantidad_maxima: int = 10) -> str:
    """Lee los correos no leídos del buzón de facturación vía IMAP.
    Retorna un resumen de los correos encontrados con sus adjuntos.
    Args:
        cantidad_maxima: Número máximo de correos a leer (default 10)
    """
    try:
        # Conexión IMAP con SSL
        mail = imaplib.IMAP4_SSL(IMAP_HOST, IMAP_PORT)
        mail.login(IMAP_USER, IMAP_PASSWORD)
        mail.select("INBOX")

        # Buscar correos no leídos
        status, messages = mail.search(None, "UNSEEN")
        if status != "OK" or not messages[0]:
            mail.logout()
            return "No hay correos no leídos en el buzón."

        email_ids = messages[0].split()
        email_ids = email_ids[:cantidad_maxima]  # Limitar cantidad

        resultados = []
        for eid in email_ids:
            status, msg_data = mail.fetch(eid, "(RFC822)")
            if status != "OK":
                continue

            msg = email.message_from_bytes(msg_data[0][1])

            # Decodificar asunto
            subject, encoding = decode_header(msg["Subject"])[0]
            if isinstance(subject, bytes):
                subject = subject.decode(encoding or "utf-8", errors="replace")

            sender = msg.get("From", "Desconocido")
            date = msg.get("Date", "Sin fecha")

            # Detectar adjuntos
            adjuntos = []
            for part in msg.walk():
                if part.get_content_disposition() == "attachment":
                    filename = part.get_filename()
                    if filename:
                        fname, enc = decode_header(filename)[0]
                        if isinstance(fname, bytes):
                            fname = fname.decode(enc or "utf-8", errors="replace")
                        adjuntos.append(fname)

            resultados.append({
                "id": eid.decode(),
                "asunto": subject,
                "remitente": sender,
                "fecha": date,
                "adjuntos": adjuntos
            })

        mail.logout()

        if not resultados:
            return "No se encontraron correos con adjuntos."

        resumen = f"Se encontraron {len(resultados)} correos no leídos:\n"
        for r in resultados:
            resumen += f"- ID:{r['id']} | De: {r['remitente']} | Asunto: {r['asunto']} | Adjuntos: {', '.join(r['adjuntos']) if r['adjuntos'] else 'Ninguno'}\n"

        # Guardar IDs en variable global para uso posterior
        global correos_pendientes
        correos_pendientes = resultados

        return resumen

    except Exception as e:
        return f"Error al leer correos: {str(e)}"

print("✅ Tool 'leer_correos' definida")

### 4.2 Tool: Extraer adjuntos de un correo

In [ ]:
@tool
def extraer_adjuntos(email_id: str) -> str:
    """Extrae los archivos adjuntos de un correo específico.
    Descomprime archivos ZIP y guarda PDF y XML en disco.
    Args:
        email_id: ID del correo del cual extraer adjuntos
    """
    try:
        mail = imaplib.IMAP4_SSL(IMAP_HOST, IMAP_PORT)
        mail.login(IMAP_USER, IMAP_PASSWORD)
        mail.select("INBOX")

        status, msg_data = mail.fetch(email_id.encode(), "(RFC822)")
        if status != "OK":
            mail.logout()
            return f"Error: No se pudo obtener el correo con ID {email_id}"

        msg = email.message_from_bytes(msg_data[0][1])

        # Crear carpeta para este correo (usar ruta absoluta)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        carpeta_correo = os.path.abspath(os.path.join(OUTPUT_DIR, f"correo_{email_id}_{timestamp}"))
        os.makedirs(carpeta_correo, exist_ok=True)

        archivos_extraidos = []

        for part in msg.walk():
            if part.get_content_disposition() != "attachment":
                continue

            filename = part.get_filename()
            if not filename:
                continue

            fname, enc = decode_header(filename)[0]
            if isinstance(fname, bytes):
                fname = fname.decode(enc or "utf-8", errors="replace")

            payload = part.get_payload(decode=True)

            # Si es ZIP, descomprimir
            if fname.lower().endswith(".zip"):
                zip_path = os.path.join(carpeta_correo, fname)
                with open(zip_path, "wb") as f:
                    f.write(payload)
                archivos_extraidos.append(f"ZIP: {zip_path}")

                # Descomprimir
                with zipfile.ZipFile(io.BytesIO(payload), "r") as zf:
                    for zipped_file in zf.namelist():
                        extracted_path = os.path.abspath(os.path.join(carpeta_correo, zipped_file))
                        # Prevenir path traversal
                        if not extracted_path.startswith(carpeta_correo):
                            continue
                        zf.extract(zipped_file, carpeta_correo)
                        archivos_extraidos.append(f"  → {zipped_file}")
            else:
                # Guardar archivo directamente (PDF, XML, etc.)
                file_path = os.path.join(carpeta_correo, fname)
                with open(file_path, "wb") as f:
                    f.write(payload)
                archivos_extraidos.append(f"{fname.split('.')[-1].upper()}: {file_path}")

        mail.logout()

        if not archivos_extraidos:
            return f"No se encontraron adjuntos en el correo {email_id}"

        return f"Archivos extraídos en '{carpeta_correo}':\n" + "\n".join(archivos_extraidos)

    except Exception as e:
        return f"Error al extraer adjuntos: {str(e)}"

print("✅ Tool 'extraer_adjuntos' definida")

### 4.3 Tool: Parsear factura electrónica (XML)

In [ ]:
@tool
def parsear_factura(ruta_carpeta: str) -> str:
    """Parsea los archivos XML de factura electrónica colombiana (UBL 2.1) en una carpeta.
    Extrae: NIT emisor, razón social, número de factura, fecha, total, y rutas de archivos.
    Args:
        ruta_carpeta: Ruta de la carpeta donde están los archivos extraídos
    """
    try:
        if not os.path.exists(ruta_carpeta):
            return f"Error: La carpeta '{ruta_carpeta}' no existe."

        # Buscar archivos XML en la carpeta
        xml_files = [f for f in os.listdir(ruta_carpeta) if f.lower().endswith(".xml")]
        pdf_files = [f for f in os.listdir(ruta_carpeta) if f.lower().endswith(".pdf")]

        if not xml_files:
            return f"No se encontraron archivos XML en '{ruta_carpeta}'"

        facturas_parseadas = []

        for xml_file in xml_files:
            xml_path = os.path.join(ruta_carpeta, xml_file)
            try:
                tree = ET.parse(xml_path)
                root = tree.getroot()

                # Detectar namespaces automáticamente
                nsmap = {}
                for event, elem in ET.iterparse(xml_path, events=["start-ns"]):
                    prefix, uri = elem
                    nsmap[prefix] = uri

                # Namespaces comunes en factura electrónica colombiana UBL 2.1
                ns = {
                    "cbc": nsmap.get("cbc", "urn:oasis:names:specification:ubl:schema:xsd:CommonBasicComponents-2"),
                    "cac": nsmap.get("cac", "urn:oasis:names:specification:ubl:schema:xsd:CommonAggregateComponents-2"),
                }

                # Extraer campos clave
                def find_text(xpath, default="N/A"):
                    elem = root.find(xpath, ns)
                    return elem.text.strip() if elem is not None and elem.text else default

                # Número de factura
                numero_factura = find_text(".//cbc:ID")

                # Fecha de emisión
                fecha = find_text(".//cbc:IssueDate")

                # Datos del emisor (proveedor)
                nit_emisor = find_text(".//cac:AccountingSupplierParty/cac:Party/cac:PartyTaxScheme/cbc:CompanyID")
                if nit_emisor == "N/A":
                    nit_emisor = find_text(".//cac:SenderParty/cac:PartyTaxScheme/cbc:CompanyID")

                razon_social = find_text(".//cac:AccountingSupplierParty/cac:Party/cac:PartyTaxScheme/cbc:RegistrationName")
                if razon_social == "N/A":
                    razon_social = find_text(".//cac:AccountingSupplierParty/cac:Party/cac:PartyLegalEntity/cbc:RegistrationName")

                # Total
                total = find_text(".//cac:LegalMonetaryTotal/cbc:PayableAmount", "0")
                if total == "0":
                    total = find_text(".//cac:LegalMonetaryTotal/cbc:TaxInclusiveAmount", "0")

                # Descripción (primer item)
                descripcion = find_text(".//cac:InvoiceLine/cac:Item/cbc:Description")
                if descripcion == "N/A":
                    descripcion = find_text(".//cac:InvoiceLine/cac:Item/cbc:Name")

                factura_info = {
                    "nit_emisor": nit_emisor,
                    "razon_social": razon_social,
                    "numero_factura": numero_factura,
                    "fecha_emision": fecha,
                    "valor_total": total,
                    "descripcion": descripcion,
                    "archivo_xml": xml_path,
                    "archivo_pdf": os.path.join(ruta_carpeta, pdf_files[0]) if pdf_files else "No encontrado",
                }
                facturas_parseadas.append(factura_info)

            except ET.ParseError as pe:
                facturas_parseadas.append({"archivo_xml": xml_path, "error": f"Error de parseo: {str(pe)}"})

        resultado = f"Se encontraron {len(xml_files)} XML y {len(pdf_files)} PDF en la carpeta.\n\n"
        for f in facturas_parseadas:
            if "error" in f:
                resultado += f"❌ {f['archivo_xml']}: {f['error']}\n"
            else:
                resultado += f"📄 Factura encontrada:\n"
                resultado += f"   NIT Emisor: {f['nit_emisor']}\n"
                resultado += f"   Razón Social: {f['razon_social']}\n"
                resultado += f"   Número Factura: {f['numero_factura']}\n"
                resultado += f"   Fecha Emisión: {f['fecha_emision']}\n"
                resultado += f"   Valor Total: {f['valor_total']}\n"
                resultado += f"   Descripción: {f['descripcion']}\n"
                resultado += f"   PDF: {f['archivo_pdf']}\n"
                resultado += f"   XML: {f['archivo_xml']}\n\n"

        return resultado

    except Exception as e:
        return f"Error al parsear factura: {str(e)}"

print("✅ Tool 'parsear_factura' definida")

## 5. Configuración del Agente LangChain

### 5.1 LLM y Tools

In [ ]:
# Inicializar LLM
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,  # Determinístico para extracción de datos
)

# Definir las herramientas del agente
tools = [leer_correos, extraer_adjuntos, parsear_factura]

# Prompt del sistema para el agente
SYSTEM_PROMPT = """Eres un agente especializado en procesamiento de facturas electrónicas colombianas.
Tu trabajo es:
1. Leer los correos no leídos del buzón de facturación
2. Extraer los archivos adjuntos (ZIP, PDF, XML) de cada correo
3. Parsear los XML de las facturas para extraer información estructurada
4. Presentar un resumen con los datos de cada factura

De cada factura debes extraer (cuando esté disponible en el XML):
- NIT del emisor
- Razón social del emisor
- Número de factura
- Fecha de emisión
- Valor total
- Descripción/concepto
- Ruta del archivo PDF
- Ruta del archivo XML

IMPORTANTE: Procesa TODOS los correos encontrados, uno por uno.
Al final, presenta un resumen consolidado en formato de tabla."""

print("✅ LLM y Prompt configurados")
print(f"   Modelo: {llm.model_name}")
print(f"   Temperatura: {llm.temperature}")
print(f"   Tools: {[t.name for t in tools]}")

### 5.2 Crear el Agente (ReAct con LangGraph)

In [ ]:
# Crear agente ReAct con LangGraph
agent_executor = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT
)

print("✅ Agente ReAct creado con LangGraph")
print(f"   Estrategia: ReAct (Reasoning + Acting)")
print(f"   Framework: langgraph.prebuilt.create_react_agent")

## 6. Ejecución del Agente

### 6.1 Disparador Manual

In [ ]:
# Variable global para almacenar resultados
correos_pendientes = []
registro_facturas = []

def ejecutar_agente():
    """Ejecuta el agente de facturación (disparador manual)."""
    print("="*60)
    print("🚀 EJECUCIÓN DEL AGENTE DE FACTURACIÓN")
    print(f"   Hora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)

    resultado = agent_executor.invoke({
        "messages": [("human", "Lee todos los correos no leídos del buzón, extrae los adjuntos de cada uno, parsea las facturas XML y dame un resumen completo con los datos de cada factura.")]
    })

    return resultado

# Ejecutar manualmente
resultado = ejecutar_agente()

### 6.2 Ver cadena de pensamiento (CoT)

In [ ]:
# Mostrar la cadena de pensamiento (Chain of Thought)
print("\n" + "="*60)
print("🧠 CADENA DE PENSAMIENTO (Chain of Thought)")
print("="*60)

messages = resultado["messages"]
for i, msg in enumerate(messages):
    tipo = msg.__class__.__name__
    if tipo == "HumanMessage":
        print(f"\n👤 Human: {msg.content[:200]}")
    elif tipo == "AIMessage":
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"\n🤖 Thought → Action: {tc['name']}({tc['args']})")
        elif msg.content:
            print(f"\n🤖 AI: {msg.content[:500]}")
    elif tipo == "ToolMessage":
        print(f"   🔧 Tool Result ({msg.name}): {msg.content[:200]}...")

# Respuesta final
final_msg = messages[-1]
print("\n" + "="*60)
print("📋 RESPUESTA FINAL DEL AGENTE:")
print("="*60)
print(final_msg.content)

### 6.3 Postprocesamiento: Extraer datos estructurados con LLM

In [ ]:
import json
import re

# Usar el LLM para estructurar la respuesta del agente en JSON
prompt_estructurar = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente que convierte texto sobre facturas electrónicas en JSON estructurado.
Extrae la información de cada factura y devuelve un JSON array con objetos que tengan estos campos:
- nit_emisor: string
- razon_social: string
- numero_factura: string
- fecha_emision: string (YYYY-MM-DD)
- valor_total: number
- descripcion: string
- ruta_pdf: string
- ruta_xml: string

Si algún campo no está disponible, usa "N/A" para strings y 0 para numbers.
Responde SOLO con el JSON, sin texto adicional."""),
    ("human", "{texto_facturas}")
])

chain_estructurar = prompt_estructurar | llm

# Estructurar la respuesta final del agente
final_msg = resultado["messages"][-1]
respuesta_json = chain_estructurar.invoke({"texto_facturas": final_msg.content})

# Limpiar markdown code blocks si el LLM los agrega
contenido = respuesta_json.content.strip()
contenido = re.sub(r'^```(?:json)?\s*', '', contenido)
contenido = re.sub(r'\s*```$', '', contenido)

try:
    facturas_data = json.loads(contenido)
    print(f"✅ Se extrajeron {len(facturas_data)} facturas estructuradas")
    print(json.dumps(facturas_data, indent=2, ensure_ascii=False))
except json.JSONDecodeError as e:
    print(f"⚠️ No se pudo parsear el JSON: {e}")
    print(contenido)
    facturas_data = []

## 7. Visualización en Tabla (DataFrame)

In [ ]:
# Crear DataFrame con los resultados
if facturas_data:
    df_facturas = pd.DataFrame(facturas_data)
    
    # Formatear para mejor visualización
    print("\n" + "="*60)
    print("📊 REGISTRO DE FACTURAS PROCESADAS")
    print("="*60)
    print(f"Total facturas: {len(df_facturas)}")
    print(f"Fecha de procesamiento: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)
    
    display(df_facturas)
else:
    print("No hay facturas para mostrar. Ejecuta el agente primero.")
    df_facturas = pd.DataFrame(columns=[
        "nit_emisor", "razon_social", "numero_factura",
        "fecha_emision", "valor_total", "descripcion",
        "ruta_pdf", "ruta_xml"
    ])

## 8. Disparador Automático (Job cada 5 minutos)

In [ ]:
import threading
import time

# Variable de control para detener el job
job_activo = False

def job_periodico(intervalo_segundos=300):
    """Ejecuta el agente periódicamente cada N segundos."""
    global job_activo, registro_facturas
    job_activo = True
    ejecucion_num = 0

    print(f"⏰ Job automático iniciado. Intervalo: {intervalo_segundos}s ({intervalo_segundos//60} min)")
    print("   Para detener: ejecutar la celda 'Detener Job'\n")

    while job_activo:
        ejecucion_num += 1
        print(f"\n{'─'*40}")
        print(f"🔄 Ejecución #{ejecucion_num} - {datetime.now().strftime('%H:%M:%S')}")
        print(f"{'─'*40}")

        try:
            res = agent_executor.invoke({
                "messages": [("human", "Lee todos los correos no leídos, extrae adjuntos, parsea las facturas y dame el resumen.")]
            })
            respuesta = res["messages"][-1].content
            print(f"   ✅ Ejecución completada")
            print(f"   Resultado: {respuesta[:200]}...")
        except Exception as e:
            print(f"   ❌ Error en ejecución: {str(e)}")

        # Esperar hasta la siguiente ejecución
        if job_activo:
            print(f"   ⏳ Próxima ejecución en {intervalo_segundos}s...")
            time.sleep(intervalo_segundos)

    print("\n⏹️ Job automático detenido.")


# Iniciar job en un hilo separado (intervalo: 300s = 5 min)
# DESCOMENTAR la siguiente línea para activar el job automático:
# thread_job = threading.Thread(target=job_periodico, args=(300,), daemon=True)
# thread_job.start()

print("💡 Para activar el job automático, descomenta las líneas anteriores y ejecuta esta celda.")
print("   El job se ejecutará cada 5 minutos en segundo plano.")

In [ ]:
# Detener Job
job_activo = False
print("⏹️ Señal de detención enviada al job.")

## 9. Listar archivos extraídos

In [ ]:
# Mostrar estructura de archivos extraídos
print("📁 Archivos extraídos:")
print("="*40)

if os.path.exists(OUTPUT_DIR):
    for carpeta in sorted(os.listdir(OUTPUT_DIR)):
        carpeta_path = os.path.join(OUTPUT_DIR, carpeta)
        if os.path.isdir(carpeta_path):
            print(f"\n📂 {carpeta}/")
            for archivo in sorted(os.listdir(carpeta_path)):
                ext = archivo.split('.')[-1].upper()
                icono = {"PDF": "📑", "XML": "📄", "ZIP": "🗜️"}.get(ext, "📎")
                print(f"   {icono} {archivo}")
else:
    print("No se han extraído archivos aún.")

## 10. Exportar tabla a CSV

In [ ]:
# Exportar resultados a CSV
if not df_facturas.empty:
    csv_path = "registro_facturas.csv"
    df_facturas.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"✅ Tabla exportada a: {csv_path}")
    print(f"   Registros: {len(df_facturas)}")
else:
    print("No hay datos para exportar.")

---
## Resumen de Componentes LangChain utilizados

| Componente | Implementación | Justificación |
|---|---|---|
| **LLM** | `ChatOpenAI(model="gpt-4o", temperature=0)` | Precisión en extracción de datos |
| **Prompt** | `ChatPromptTemplate` + `PromptTemplate` (ReAct) | Instrucciones claras al agente |
| **Tools** | `leer_correos`, `extraer_adjuntos`, `parsear_factura` | Capacidades IMAP + filesystem |
| **Agent** | `create_react_agent` + `AgentExecutor` | Razonamiento paso a paso (CoT) |
| **Chain** | `prompt | llm` (LCEL) para post-procesamiento | Estructurar salida en JSON |

**Estrategia de razonamiento:** ReAct (Reasoning + Acting) con Zero-shot CoT visible en el log `verbose=True`.